# Tensile data received so far — analysis (2026-10)

Analysis of the latest delivery, `raw_data/10_26_26/` (identical to 10_16_26): the 12 formulations of the August DoE, all **0x recycled
(cycle 0)**, 44 specimens. Extends `data_processing_v3.ipynb` (September delivery, 30 specimens) with the TPU formulations and adds the
statistics used in the October slides: outlier screen, replicate CV, pair plot, ANOVA (formulation means, specimen level, lack of fit),
GP partial dependence, interactions and Sobol indices, and trade-offs.

Logic in `utils/instron.py` (parsing) and `utils/analysis.py` (statistics); figures in `utils/plots.py`. Run in the `ml_gp_env` kernel
(statsmodels). **Nothing is written by default.**

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "utils").exists() and (ROOT.parent / "utils").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import config, io, doe, plots

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
plt.rcParams["figure.dpi"] = 110
p = config.paths(ROOT)

In [ ]:
from utils import instron, analysis as A

RAW = p.root / "raw_data" / "10_26_26"
design = pd.read_csv(p.doe / "doe_formulations.csv")
plan = pd.read_csv(p.doe / "doe_measurement_plan.csv")
cat = instron.catalogue(io.extract_zip(RAW))
print(f"{len(cat)} specimens · {cat['formulation'].nunique()} formulations · recycled_x {sorted(cat['recycled_x'].unique())}")

---
## 1 — Coverage against the first DoE

Same steps as v3 (`coverage`, `compare_to_plan`), plus the per-level count: the lab's "0x recycled" = first extrusion = plan level 1×.

In [ ]:
cov = instron.coverage(design, cat)
cat["plan_level"] = cat["recycled_x"] + 1
got = cat.groupby(["formulation", "plan_level"]).size().rename("specimens").reset_index()
m = plan.merge(got, left_on=["formulation", "recycle_level"], right_on=["formulation", "plan_level"], how="left")
m["specimens"] = m["specimens"].fillna(0).astype(int)
by_level = m.groupby("recycle_level").agg(planned=("run", "count"), delivered=("specimens", lambda s: int((s > 0).sum())),
                                          specimens=("specimens", "sum"))
by_level

---
## 2 — Parse, set aside, screen

* **f11_s3** — failed test: no load recorded (UTS 0.12 MPa, modulus fit R² 0.001).
* **f08_s1** — the original f08 specimen (Sept): 119 % strain vs 7 % for the three remade specimens (`f08-new`); the remake fits the TPU
  trend with PAL. Set aside pending the lab's confirmation.

In [ ]:
SAVE_CSV = False                      # True -> write the per-specimen table to processed_data/tensile_doe/
CSV_NAME = "tensile_oct_summary"

curves, raw = instron.load_delivery(cat)
summary = instron.attach_doe(raw, design)
DROP = {"f11_s3": "failed test: no load recorded (UTS 0.12 MPa, modulus fit R² 0.001)",
        "f08_s1": "superseded by the f08 remake (f08-new): 119 % strain vs 7 % for the 3 remade specimens"}
kept, dropped = A.drop_specimens(summary, DROP)
if SAVE_CSV:
    summary.to_csv(io.save_path(p.tensile_doe, CSV_NAME, ext=".csv"), index=False)
print(f"{len(kept)} specimens kept")
dropped

In [ ]:
flags = A.pooled_outliers(kept)          # pooled log residuals from the formulation median, IQR fences
flags[flags["n_flags"] > 0]

In [ ]:
cv = A.cv_table(kept)
print("median CV %:", cv[list(A.RESPONSES)].median().round(1).to_dict())
cv.round(1)

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_oct_cv"

plots.cv_heatmap(cv, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

---
## 3 — Curves and properties

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_oct_curves"

plots.tensile_curves_by_polymer(curves, kept, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_oct_props"

plots.props_by_formulation_log(kept, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_oct_props_vs_factors"

plots.props_vs_factors_log(kept, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_oct_props_vs_factors_connected"

# same, with each polymer's means joined in order of the x factor (guide for the eye: the other factor varies too)
plots.props_vs_factors_log(kept, connect=True, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

---
## 4 — Pair plot

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_oct_pairs"

plots.pair_plot_specimens(kept, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

Correlation between the responses (Pearson on log values, specimens), pooled and per polymer. The pooled matrix mostly
reflects the differences between polymers (TPU weak and stretchy, PP / HDPE strong and brittle); within a polymer the sign can flip.

In [ ]:
corrs = A.response_correlation(kept, polymers=config.DOE.polymers)
{k: c.round(2) for k, c in corrs.items()}

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_oct_response_correlation"

plots.response_correlation(corrs, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

---
## 5 — ANOVA

Lack of fit first: if the formulations differ by more than replicate scatter explains, the specimen-level p-values are optimistic and the
**formulation means** are the unit for inference (main-effects model, 12 means).

In [ ]:
A.lack_of_fit(kept).round(4)

In [ ]:
an_means = A.anova_means(kept)
an_means.round(4)

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_oct_anova"

plots.anova_effects(an_means, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

In [ ]:
A.anova_tables(kept).round(4)            # specimen level, polymer x (PAL + temperature): effect sizes, optimistic p

---
## 6 — Partial dependence (GP)

GP on log(response), one-hot polymer + scaled PAL and temperature (Matern 5/2, ARD, white noise), fitted on the specimens; partial dependence
per polymer with a 5–95 % band from joint posterior samples. About 2 min.

In [ ]:
pds = {r: A.gp_partial_dependence(kept, r, config.DOE.polymers) for r in A.RESPONSES}
{r: d.attrs["kernel"] for r, d in pds.items()}

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_oct_partial_dependence"

plots.partial_dependence_grid(pds, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

---
## 7 — Interactions and Sobol indices

Two views of main effects (1st order) and interactions (2nd, 3rd order); all runs are at cycle 0, so nothing can be said about recycling yet.

* **Means, nested F tests:** each two-way interaction added on its own to the additive means model (5–6 residual df; all at once leaves 3).
  Where polymer × PAL is significant, the PAL effect is estimated per polymer.
* **GP Sobol indices:** the PD GP decomposed exactly on a polymer × PAL × temperature grid (inputs uniform over their ranges): share of the
  variance of log(response) per main effect and interaction; total index = main effect + all its interactions. 5–95 % over posterior draws of
  the latent function (specimen noise left out). *Within a polymer* the shares are relative to that polymer's own variation, which can be
  small (e.g. PP modulus). About 2 min.

In [ ]:
A.interaction_tests(kept).round(4)

In [ ]:
A.anova_means(kept, rhs="C(polymer) * pal + temp").round(4)   # one PAL slope per polymer

In [ ]:
A.pal_effect_by_polymer(kept).round(3)                        # % change per +10 PAL points, 95 % CI

In [ ]:
sob = pd.concat([A.gp_sobol(kept, r, config.DOE.polymers) for r in A.RESPONSES], ignore_index=True)
(sob[sob["scope"] == "all"].assign(order=lambda d: d["order"].astype(str))
 .pivot_table(index=["order", "term"], columns="response", values="median", sort=False)[list(A.RESPONSES)].round(3))

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_oct_sobol"

plots.sobol_indices(sob, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

---
## 8 — Trade-offs

In [ ]:
SAVE_FIG = False                      # True -> write the figure to processed_data/figures/
FIG_NAME = "tensile_oct_tradeoffs"

plots.tradeoffs(kept, save=io.save_path(p.figures, FIG_NAME, SAVE_FIG, ".png"))

---
## Reading the result (2026-10-28)

* **Coverage:** 12 / 36 measurements of the first DoE (all at 1× = cycle 0); 5× and 10× missing; no rheology.
* **Scatter:** median replicate CV — UTS 12 %, modulus 16 %, strain 27 %, toughness 40 %; 3 specimens flagged by the pooled screen, none removed.
* **ANOVA (formulation means):** polymer dominates UTS, modulus, strain (partial η² 0.81–0.94, p ≤ 0.003); PAL lowers UTS (p < 0.001) and
  toughness (p = 0.03); temperature not resolved (UTS p = 0.08). Lack of fit p ≤ 0.012 for every response.
* **Interactions:** polymer × PAL is the only one — means F test p = 0.003 (modulus), 0.017 (strain), 0.028 (toughness), 0.27 (UTS); it is
  TPU: per +10 PAL points its modulus rises 76 % and strain / toughness fall 66 / 77 %, while PP and HDPE change little. No polymer ×
  temperature or PAL × temperature effect (all p > 0.05; closest PAL × temperature on strain, p = 0.07); 3rd order ≈ 0. With per-polymer PAL slopes, PAL is significant for all four responses.
* **Sobol (GP, log scale):** 1st order polymer 0.36–0.92, PAL 0.01–0.32, temperature ≤ 0.02; 2nd order polymer × PAL 0.05 (UTS),
  0.06 (modulus), 0.16 (strain), 0.30 (toughness), all others ≤ 0.01. Within PP and HDPE, PAL vs temperature is not resolved (wide intervals).
* **Partial dependence:** UTS falls with PAL in all polymers; TPU strain and toughness collapse above ~30 % PAL while its modulus rises;
  temperature curves nearly flat.
* **Trade-offs:** no strength–toughness trade-off within PP / HDPE; across polymers stiffness trades against ductility.